In [10]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import joblib

from scipy.sparse import hstack

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report,f1_score

from xgboost import XGBClassifier

from src.text_preprocess import clean_text_ml

In [2]:
import mlflow
import mlflow.sklearn
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

from src.config_mlflow import setup_mlflow

setup_mlflow()

MLflow connected successfully


In [3]:
mlflow.set_experiment("ML_Model_Priority_Classification")

<Experiment: artifact_location='/Users/rishabh/Desktop/Customer Support Intelligence/mlruns/5', creation_time=1781082327619, experiment_id='5', last_update_time=1781082327619, lifecycle_stage='active', name='ML_Model_Priority_Classification', tags={}, trace_location=None, workspace='default'>

In [4]:
df=pd.read_csv('../data/ticket_classification.csv')
df=df.dropna()

In [5]:
df['text'] = df['subject'].apply(clean_text_ml)+" "+df['body'].apply(clean_text_ml)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 account disruption dear customer support team , writing report significant problem centralized account management portal , currently appears offline . outage blocking access account setting , leading substantial inconvenience . attempted log multiple time using different browser device , issue persists . could please provide update outage status estimated time resolution ? also , alternative way access manage account downtime ?


In [6]:
tfidf = joblib.load("../models/tfidf_type.pkl")

X_text = tfidf.transform(df['text'])

In [7]:
queue_encoder = joblib.load('../models/label_encoder.pkl')
type_encoder = joblib.load('../models/type_label_encoder.pkl')
priority_encoder = LabelEncoder()

df['queue_encoded'] = queue_encoder.fit_transform(df['queue'])

df['type_encoded'] = type_encoder.fit_transform(df['type'])

y = priority_encoder.fit_transform(df['priority'])

joblib.dump(
    priority_encoder,
    "../models/priority_label_encoder.pkl"
)

['../models/priority_label_encoder.pkl']

In [8]:
extra_features = np.column_stack([
    df['queue_encoded'],
    df['type_encoded']
])

X_final = hstack([
    X_text,
    extra_features
])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
model_name = "XGBoost_Classifier"

with mlflow.start_run(run_name=model_name):
    model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    max_depth=10,
    n_jobs=-1,
    n_estimators=100,
    learning_rate=0.2,
    gamma=0.1,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)

    mlflow.log_param("max_depth", 10)
    mlflow.log_param("vectorizer", "TF-IDF")
    mlflow.log_param("learning_rate", 0.2)
    mlflow.log_param("eval_metric", "mlogloss")
    mlflow.log_param("gamma", 0.1)
    mlflow.log_param("n_estimators", 100)
    
    # Log all metrics to MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    
    # Save model artifact
    mlflow.sklearn.log_model(model, artifact_path=model_name)
    print("Accuracy:",
                accuracy_score(y_test, predictions))

    print("F1 Score:",
        f1_score(y_test, predictions, average='macro'))

    print(classification_report(y_test, predictions))

2026/06/10 15:03:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 15:03:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.7006498781478473
F1 Score: 0.6761157760421703
              precision    recall  f1-score   support

           0       0.74      0.73      0.73      1891
           1       0.83      0.45      0.58      1007
           2       0.65      0.80      0.71      2026

    accuracy                           0.70      4924
   macro avg       0.74      0.66      0.68      4924
weighted avg       0.72      0.70      0.69      4924

🏃 View run XGBoost_Classifier at: http://127.0.0.1:5000/#/experiments/5/runs/ccdfd0e03cee4bdd9ebb0f697051679b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [13]:
model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    max_depth=10,
    n_jobs=-1,
    n_estimators=100
    )
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)


In [14]:
print("Accuracy:",
                accuracy_score(y_test, predictions))

print("F1 Score:",
        f1_score(y_test, predictions, average='macro'))

print(classification_report(y_test, predictions))

Accuracy: 0.7128350934199837
F1 Score: 0.69207078278655
              precision    recall  f1-score   support

           0       0.75      0.75      0.75      1891
           1       0.83      0.48      0.61      1007
           2       0.66      0.80      0.72      2026

    accuracy                           0.71      4924
   macro avg       0.74      0.67      0.69      4924
weighted avg       0.73      0.71      0.71      4924



In [15]:
joblib.dump(
    model,
    "../models/priority_xgb.pkl"
)

['../models/priority_xgb.pkl']